# EXP-29: Data Augmentation — Synonym + Sentence Shuffle + Dropout

**Competition:** SPR 2026 — Mammography Report Classification  
**Base:** EXP-24 (0.81021 LB) — Specialist Ensemble  
**Strategy:** Data augmentation para classes minoritárias  

**Técnicas de augmentation:**
1. **Sinônimos clínicos**: Substitui termos médicos por sinônimos (nódulo↔lesão nodular, espiculado↔com espículas)
2. **Sentence shuffle**: Reordena sentenças dentro da seção de achados
3. **Sentence dropout**: Remove aleatoriamente 1 sentença (simula laudos mais curtos)
4. **Augmentation seletivo**: Só augmenta classes com < 500 amostras para balancear

**Hipótese:** Classes raras (0, 3, 5, 6) sofrem com poucos exemplos. Augmentation pode ajudar o modelo a generalizar melhor nessas classes.  
**Runtime:** CPU only

In [ ]:
import os, re, hashlib, warnings, time, random
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.linear_model import LogisticRegression, RidgeClassifier
from sklearn.pipeline import FeatureUnion
from sklearn.model_selection import GroupKFold
from sklearn.metrics import f1_score, classification_report
from scipy.sparse import hstack, csr_matrix
import lightgbm as lgb
warnings.filterwarnings('ignore')

TRAIN_PATH = '/kaggle/input/competitions/spr-2026-mammography-report-classification/train.csv'
TEST_PATH  = '/kaggle/input/competitions/spr-2026-mammography-report-classification/test.csv'

train_orig = pd.read_csv(TRAIN_PATH)
test  = pd.read_csv(TEST_PATH)

N_FOLDS = 5
N_CLASSES = 7
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

print(f'Train: {train_orig.shape}, Test: {test.shape}')
print(f'Class distribution:\n{train_orig["target"].value_counts().sort_index()}')

In [ ]:
# =============================================================================
# Cell 2: Data Augmentation Functions
# =============================================================================

# Clinical synonym dictionary (Portuguese mammography terms)
SYNONYMS = {
    'nódulo': ['lesão nodular', 'formação nodular', 'imagem nodular'],
    'nódulos': ['lesões nodulares', 'formações nodulares'],
    'espiculado': ['com espículas', 'de contornos espiculados', 'espiculada'],
    'calcificações': ['microcalcificações', 'focos de calcificação'],
    'assimetria': ['área de assimetria', 'assimetria focal'],
    'distorção arquitetural': ['distorção do parênquima', 'alteração arquitetural'],
    'linfonodo': ['linfonodo axilar', 'gânglio linfático'],
    'pele': ['tegumento', 'revestimento cutâneo'],
    'mama direita': ['md', 'mama d'],
    'mama esquerda': ['me', 'mama e'],
    'heterogêneo': ['heterogênea', 'de ecotextura heterogênea'],
    'irregular': ['de contornos irregulares', 'com margens irregulares'],
    'oval': ['ovalado', 'de formato oval'],
    'redondo': ['arredondado', 'de formato arredondado'],
    'denso': ['densa', 'de alta densidade'],
    'suspeita': ['suspeito', 'com características suspeitas'],
    'benigno': ['benigna', 'de aspecto benigno', 'provavelmente benigno'],
}

def augment_synonym(text, n_replacements=2):
    """Replace random clinical terms with synonyms"""
    text_lower = text.lower()
    replaceable = [(k, v) for k, v in SYNONYMS.items() if k in text_lower]
    if not replaceable:
        return text
    
    n = min(n_replacements, len(replaceable))
    chosen = random.sample(replaceable, n)
    result = text
    for original, syns in chosen:
        replacement = random.choice(syns)
        # Case-insensitive replacement (first occurrence only)
        pattern = re.compile(re.escape(original), re.IGNORECASE)
        result = pattern.sub(replacement, result, count=1)
    return result

def augment_sentence_shuffle(text):
    """Shuffle sentences within the text (preserving section headers)"""
    lines = [l.strip() for l in text.split('\n') if l.strip()]
    if len(lines) <= 2:
        return text
    
    # Keep first line (usually header) and last line, shuffle middle
    header = [lines[0]] if re.match(r'^(achados|técnica|indicação)', lines[0].lower()) else []
    footer_keywords = ['impressão', 'conclusão', 'opinião', 'recomend']
    
    body = []
    footer = []
    in_footer = False
    for line in (lines[len(header):]):
        if any(kw in line.lower() for kw in footer_keywords):
            in_footer = True
        if in_footer:
            footer.append(line)
        else:
            body.append(line)
    
    if len(body) > 1:
        random.shuffle(body)
    
    return '\n'.join(header + body + footer)

def augment_sentence_dropout(text, drop_prob=0.15):
    """Randomly drop sentences (simulates shorter/incomplete reports)"""
    lines = [l.strip() for l in text.split('\n') if l.strip()]
    if len(lines) <= 3:
        return text  # Don't drop from very short reports
    
    # Don't drop section headers or conclusions
    protected_keywords = ['achados', 'técnica', 'indicação', 'impressão', 'conclusão', 'opinião', 'bi-rads', 'categoria']
    
    kept = []
    for line in lines:
        is_protected = any(kw in line.lower() for kw in protected_keywords)
        if is_protected or random.random() > drop_prob:
            kept.append(line)
    
    return '\n'.join(kept) if kept else text

def augment_report(report, method='random'):
    """Apply a random augmentation method"""
    if method == 'random':
        method = random.choice(['synonym', 'shuffle', 'dropout', 'synonym+dropout'])
    
    if method == 'synonym':
        return augment_synonym(report, n_replacements=2)
    elif method == 'shuffle':
        return augment_sentence_shuffle(report)
    elif method == 'dropout':
        return augment_sentence_dropout(report, drop_prob=0.15)
    elif method == 'synonym+dropout':
        return augment_sentence_dropout(augment_synonym(report, 1), 0.10)
    return report

# Quick test
sample = train_orig.iloc[0]['report']
print("Original (first 200 chars):")
print(sample[:200])
print("\nAugmented (synonym):")
print(augment_synonym(sample)[:200])

In [ ]:
# =============================================================================
# Cell 3: Generate augmented data for minority classes
# =============================================================================

TARGET_SAMPLES = 500  # Augment classes below this threshold
MAX_AUG_RATIO = 3     # Maximum augmentation multiplier per sample

class_counts = train_orig['target'].value_counts().sort_index()
print("Original class counts:")
print(class_counts)

augmented_rows = []
for cls in range(N_CLASSES):
    count = class_counts.get(cls, 0)
    if count >= TARGET_SAMPLES:
        print(f"Class {cls}: {count} samples, no augmentation needed")
        continue
    
    needed = min(TARGET_SAMPLES - count, count * MAX_AUG_RATIO)  # Cap augmentation
    cls_data = train_orig[train_orig['target'] == cls]
    
    print(f"Class {cls}: {count} samples, generating {needed} augmented samples")
    
    for i in range(needed):
        # Sample a random row from this class
        row = cls_data.sample(1, random_state=SEED + i).iloc[0]
        aug_report = augment_report(row['report'], method='random')
        augmented_rows.append({
            'report': aug_report,
            'target': cls,
            'ID': f"aug_{cls}_{i}"
        })

if augmented_rows:
    aug_df = pd.DataFrame(augmented_rows)
    train = pd.concat([train_orig, aug_df[['ID', 'report', 'target']]], ignore_index=True)
    print(f"\nTotal train after augmentation: {len(train)} (+{len(aug_df)} augmented)")
else:
    train = train_orig.copy()
    print("\nNo augmentation needed")

print(f"\nNew class distribution:\n{train['target'].value_counts().sort_index()}")

In [ ]:
# =============================================================================
# Cell 4: Text Preprocessing (igual EXP-24)
# =============================================================================

def clean_achados(s):
    if pd.isna(s): return ""
    s = str(s).strip().lower()
    s = s.replace("\r\n", "\n").replace("\r", "\n")
    s = re.sub(r"[ \t]+", " ", s)
    s = re.sub(r"\n{2,}", "\n", s)
    match = re.search(r'achados[:\s]*(.*?)(?:impress[aã]o|conclus[aã]o|an[aá]lise comparativa|opini[aã]o|$)', s, flags=re.DOTALL)
    if match: s = match.group(1).strip()
    s = re.sub(r'[0-9]+,[0-9]+', 'NUM', s)
    s = re.sub(r'[0-9]+', 'NUM', s)
    return s

def clean_full(s):
    if pd.isna(s): return ""
    s = str(s).strip().lower()
    s = s.replace("\r\n", "\n").replace("\r", "\n")
    s = re.sub(r"[ \t]+", " ", s)
    s = re.sub(r'[0-9]+,[0-9]+', 'NUM', s)
    s = re.sub(r'[0-9]+', 'NUM', s)
    return s

def extract_conclusion(s):
    if pd.isna(s): return ""
    s = str(s).strip().lower()
    s = s.replace("\r\n", "\n").replace("\r", "\n")
    match = re.search(r'(?:impress[aã]o|conclus[aã]o|opini[aã]o)[:\s]*(.*?)(?:an[aá]lise comparativa|recomenda|$)', s, flags=re.DOTALL)
    if match:
        c = match.group(1).strip()
        c = re.sub(r'[0-9]+,[0-9]+', 'NUM', c)
        c = re.sub(r'[0-9]+', 'NUM', c)
        return c
    return ""

def stable_hash(s):
    return hashlib.md5(str(s).encode("utf-8")).hexdigest()

for df in [train, test]:
    df["achados"]    = df["report"].apply(clean_achados)
    df["full"]       = df["report"].apply(clean_full)
    df["conclusion"] = df["report"].apply(extract_conclusion)

# Groups: augmented samples get unique groups to avoid leakage
train["group"] = train.apply(
    lambda row: stable_hash(row['report']) if not str(row['ID']).startswith('aug_') 
    else f"aug_{row.name}", axis=1
)

y = train["target"].astype(int).values
groups = train["group"].values

print(f'Achados found: {(train["achados"] != "").sum()} / {len(train)}')
print(f'Conclusion found: {(train["conclusion"] != "").sum()} / {len(train)}')

In [ ]:
# =============================================================================
# Cell 5: Dense Features (igual EXP-24)
# =============================================================================

def extract_birads_number(text):
    match = re.search(r'(?:bi-?rads|categoria)\s*:?\s*(\d)', text)
    return int(match.group(1)) if match else -1

def extract_dense_features(df):
    feat = pd.DataFrame(index=df.index)
    text_col = df['report'].fillna('').astype(str).str.lower()
    
    feat['report_length']   = text_col.apply(len)
    feat['has_measurement'] = text_col.str.contains(r'\b(?:cm|mm|medindo)\b', regex=True).astype(int)
    feat['has_spiculation'] = text_col.str.contains(r'espiculad', regex=True).astype(int)
    feat['has_distortion']  = text_col.str.contains(r'distor[cç][aã]o arquitetural', regex=True).astype(int)
    feat['has_biopsy']      = text_col.str.contains(r'biopsy|bi[oó]psia|resultado de cine|carcinoma', regex=True).astype(int)
    feat['word_count']        = text_col.str.split().str.len().fillna(0).astype(int)
    feat['line_count']        = text_col.str.count('\n') + 1
    feat['achados_length']    = df['achados'].str.len() if 'achados' in df.columns else 0
    feat['conclusion_length'] = df['conclusion'].str.len() if 'conclusion' in df.columns else 0
    feat['has_nodule']        = text_col.str.contains(r'n[oó]dulo', regex=True).astype(int)
    feat['has_calcification'] = text_col.str.contains(r'calcifica[cç]', regex=True).astype(int)
    feat['has_microcalc']     = text_col.str.contains(r'microcalcifica', regex=True).astype(int)
    feat['has_asymmetry']     = text_col.str.contains(r'assimetria', regex=True).astype(int)
    feat['has_lymphnode']     = text_col.str.contains(r'linfonodo|axilar', regex=True).astype(int)
    feat['has_suspicious']    = text_col.str.contains(r'suspeit[oa]|malign[oa]', regex=True).astype(int)
    feat['has_benign']        = text_col.str.contains(r'benign[oa]|cisto[s]?\b', regex=True).astype(int)
    feat['has_gross_calc']    = text_col.str.contains(r'calcifica[cç][aã]o grosseira|calcifica[cç][aã]o vascular', regex=True).astype(int)
    feat['has_nodulo_espic']  = text_col.str.contains(r'n[oó]dulo\s+espiculad', regex=True).astype(int)
    feat['has_heterogeneo']   = text_col.str.contains(r'heterog[eê]ne', regex=True).astype(int)
    feat['has_irregular']     = text_col.str.contains(r'irregular', regex=True).astype(int)
    feat['has_bilateral']     = text_col.str.contains(r'bilateral', regex=True).astype(int)
    feat['has_birads_explicit'] = text_col.str.contains(r'bi-?rads|categoria\s*\d', regex=True).astype(int)
    feat['birads_mentioned']    = text_col.apply(extract_birads_number)
    feat['risk_score'] = (
        feat['has_spiculation'] * 3 + feat['has_distortion'] * 3 +
        feat['has_nodulo_espic'] * 4 + feat['has_biopsy'] * 5 +
        feat['has_suspicious'] * 3 + feat['has_irregular'] * 2 +
        feat['has_microcalc'] * 2 + feat['has_asymmetry'] * 2 -
        feat['has_benign'] * 2 - feat['has_gross_calc'] * 2
    )
    return csr_matrix(feat.values.astype(np.float32))

X_train_dense = extract_dense_features(train)
X_test_dense  = extract_dense_features(test)
print(f'Dense features: {X_train_dense.shape[1]}')

In [ ]:
# =============================================================================
# Cell 6: TF-IDF Features (igual EXP-24)
# =============================================================================

print("Building TF-IDF features...")

tfidf_A = FeatureUnion([
    ("word", TfidfVectorizer(ngram_range=(1, 3), min_df=3, max_df=0.95, sublinear_tf=True)),
    ("char", TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=3, max_df=0.95,
                             sublinear_tf=True, max_features=80000))
])
X_train_A = tfidf_A.fit_transform(train["achados"].values)
X_test_A  = tfidf_A.transform(test["achados"].values)

tfidf_F = FeatureUnion([
    ("word", TfidfVectorizer(ngram_range=(1, 3), min_df=3, max_df=0.95, sublinear_tf=True)),
    ("char", TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=3, max_df=0.95,
                             sublinear_tf=True, max_features=80000))
])
X_train_F = tfidf_F.fit_transform(train["full"].values)
X_test_F  = tfidf_F.transform(test["full"].values)

tfidf_F2 = FeatureUnion([
    ("word", TfidfVectorizer(ngram_range=(1, 3), min_df=3, max_df=0.95, sublinear_tf=True)),
    ("char", TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 6), min_df=3, max_df=0.95,
                             sublinear_tf=True, max_features=100000))
])
X_train_F2 = tfidf_F2.fit_transform(train["full"].values)
X_test_F2  = tfidf_F2.transform(test["full"].values)

tfidf_C = TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_df=0.95, sublinear_tf=True)
X_train_C = tfidf_C.fit_transform(train["conclusion"].values)
X_test_C  = tfidf_C.transform(test["conclusion"].values)

X_train_lgb = hstack([X_train_A, X_train_dense]).tocsr()
X_test_lgb  = hstack([X_test_A,  X_test_dense]).tocsr()

X_train_full_dense = hstack([X_train_F, X_train_C, X_train_dense]).tocsr()
X_test_full_dense  = hstack([X_test_F,  X_test_C, X_test_dense]).tocsr()

print(f"SVC-A: {X_train_A.shape[1]}, SVC-F: {X_train_F.shape[1]}, SVC-F2: {X_train_F2.shape[1]}")
print(f"Conclusion: {X_train_C.shape[1]}, LGB: {X_train_lgb.shape[1]}, Full+Dense: {X_train_full_dense.shape[1]}")

In [ ]:
# =============================================================================
# Cell 7: Level 1 — 7 Base Models (igual EXP-24)
# =============================================================================

def softmax(x):
    e = np.exp(x - x.max(axis=1, keepdims=True))
    return e / e.sum(axis=1, keepdims=True)

gkf = GroupKFold(n_splits=N_FOLDS)
folds = list(gkf.split(X_train_A, y, groups))

L1_MODELS = [
    ('svc_A', lambda: CalibratedClassifierCV(
        LinearSVC(class_weight="balanced", random_state=42, max_iter=10000), cv=3, method='sigmoid'),
     X_train_A, X_test_A),
    ('svc_F', lambda: CalibratedClassifierCV(
        LinearSVC(class_weight="balanced", random_state=42, max_iter=10000), cv=3, method='sigmoid'),
     X_train_F, X_test_F),
    ('svc_F2', lambda: CalibratedClassifierCV(
        LinearSVC(class_weight="balanced", random_state=42, max_iter=10000), cv=3, method='sigmoid'),
     X_train_F2, X_test_F2),
    ('lgb1', lambda: lgb.LGBMClassifier(
        class_weight='balanced', n_estimators=300, learning_rate=0.05,
        max_depth=6, random_state=42, n_jobs=-1, verbose=-1),
     X_train_lgb, X_test_lgb),
    ('lgb2', lambda: lgb.LGBMClassifier(
        class_weight='balanced', n_estimators=500, learning_rate=0.03,
        max_depth=7, num_leaves=63, min_child_samples=20,
        subsample=0.8, colsample_bytree=0.6,
        random_state=123, n_jobs=-1, verbose=-1),
     X_train_lgb, X_test_lgb),
    ('lr', lambda: LogisticRegression(
        class_weight='balanced', C=1.0, max_iter=5000,
        solver='lbfgs', multi_class='multinomial', random_state=42, n_jobs=-1),
     X_train_full_dense, X_test_full_dense),
    ('ridge', lambda: RidgeClassifier(class_weight='balanced', alpha=1.0),
     X_train_F, X_test_F),
]

oof_probas = {}
test_probas = {}

t0 = time.time()
for name, model_fn, X_tr, X_te in L1_MODELS:
    print(f"\n--- Level 1: {name} ---")
    oof = np.zeros((len(train), N_CLASSES), dtype=np.float64)
    te_acc = np.zeros((len(test), N_CLASSES), dtype=np.float64)
    for fi, (tr_idx, va_idx) in enumerate(folds):
        m = model_fn()
        m.fit(X_tr[tr_idx], y[tr_idx])
        if hasattr(m, 'predict_proba'):
            oof[va_idx] = m.predict_proba(X_tr[va_idx])
            te_acc += m.predict_proba(X_te) / N_FOLDS
        else:
            oof[va_idx] = softmax(m.decision_function(X_tr[va_idx]))
            te_acc += softmax(m.decision_function(X_te)) / N_FOLDS
        fold_f1 = f1_score(y[va_idx], np.argmax(oof[va_idx], axis=1), average='macro')
        print(f"  Fold {fi}: F1={fold_f1:.4f}")
    oof_f1 = f1_score(y, np.argmax(oof, axis=1), average='macro')
    print(f"  >> OOF F1: {oof_f1:.4f}")
    oof_probas[name] = oof
    test_probas[name] = te_acc

print(f"\nLevel 1 done: {len(L1_MODELS)} models in {time.time()-t0:.0f}s")

In [ ]:
# =============================================================================
# Cell 8: Level 2 — Meta-Learner (igual EXP-24)
# =============================================================================

model_names = list(oof_probas.keys())
X_meta_train = np.hstack([oof_probas[name] for name in model_names])
X_meta_test  = np.hstack([test_probas[name] for name in model_names])
print(f"Meta-features: {X_meta_train.shape[1]}")

meta_oof = np.zeros((len(train), N_CLASSES), dtype=np.float64)
meta_test_acc = np.zeros((len(test), N_CLASSES), dtype=np.float64)
for fi, (tr_idx, va_idx) in enumerate(folds):
    meta_lr = LogisticRegression(class_weight='balanced', C=1.0, max_iter=5000,
                                 solver='lbfgs', multi_class='multinomial', random_state=42)
    meta_lr.fit(X_meta_train[tr_idx], y[tr_idx])
    meta_oof[va_idx] = meta_lr.predict_proba(X_meta_train[va_idx])
    meta_test_acc += meta_lr.predict_proba(X_meta_test) / N_FOLDS
    fold_f1 = f1_score(y[va_idx], np.argmax(meta_oof[va_idx], axis=1), average='macro')
    print(f"Meta Fold {fi}: F1={fold_f1:.4f}")

meta_f1 = f1_score(y, np.argmax(meta_oof, axis=1), average='macro')
print(f"\nMeta-learner OOF: {meta_f1:.4f}")

# V12-style weighted avg
oof_svc_ens = 0.25 * oof_probas['svc_A'] + 0.40 * oof_probas['svc_F'] + 0.35 * oof_probas['svc_F2']
oof_v12 = 0.70 * oof_svc_ens + 0.30 * oof_probas['lgb1']
v12_f1 = f1_score(y, np.argmax(oof_v12, axis=1), average='macro')
print(f"V12-style OOF: {v12_f1:.4f}")

# Hybrid
oof_hybrid = 0.5 * meta_oof + 0.5 * oof_v12
hybrid_f1 = f1_score(y, np.argmax(oof_hybrid, axis=1), average='macro')
print(f"Hybrid OOF: {hybrid_f1:.4f}")

# Select best
results = {'meta': (meta_f1, meta_oof, meta_test_acc)}
te_svc_ens = 0.25 * test_probas['svc_A'] + 0.40 * test_probas['svc_F'] + 0.35 * test_probas['svc_F2']
te_v12 = 0.70 * te_svc_ens + 0.30 * test_probas['lgb1']
results['v12'] = (v12_f1, oof_v12, te_v12)
results['hybrid'] = (hybrid_f1, oof_hybrid, 0.5 * meta_test_acc + 0.5 * te_v12)

best = max(results, key=lambda k: results[k][0])
final_score, oof_blend, te_blend = results[best]
print(f"\nBest: {best} (F1={final_score:.4f})")

In [ ]:
# =============================================================================
# Cell 9: Threshold Search (igual EXP-24)
# =============================================================================

threshold_classes = [6, 5, 4, 3, 1, 0]
threshold_range = np.arange(0.05, 0.55, 0.01)

def apply_thresholds(proba, thresholds):
    preds = np.argmax(proba, axis=1).copy()
    for cls in threshold_classes:
        if cls in thresholds:
            mask = proba[:, cls] > thresholds[cls]
            for higher_cls in threshold_classes:
                if higher_cls == cls: break
                if higher_cls in thresholds:
                    mask = mask & (preds != higher_cls)
            preds[mask] = cls
    return preds

baseline_preds = np.argmax(oof_blend, axis=1)
baseline_f1 = f1_score(y, baseline_preds, average='macro')
print(f'Baseline OOF (no thresholds): {baseline_f1:.4f}')

best_thresholds = {}
current_f1 = baseline_f1
for cls in threshold_classes:
    best_cls_f1 = current_f1
    best_cls_thr = None
    for thr in threshold_range:
        trial = {**best_thresholds, cls: thr}
        trial_preds = apply_thresholds(oof_blend, trial)
        trial_f1 = f1_score(y, trial_preds, average='macro')
        if trial_f1 > best_cls_f1:
            best_cls_f1 = trial_f1
            best_cls_thr = thr
    if best_cls_thr is not None:
        best_thresholds[cls] = best_cls_thr
        current_f1 = best_cls_f1
        print(f'Class {cls}: threshold={best_cls_thr:.2f}, F1={best_cls_f1:.4f}')
    else:
        print(f'Class {cls}: no improvement')

final_oof_preds = apply_thresholds(oof_blend, best_thresholds)
final_oof_f1 = f1_score(y, final_oof_preds, average='macro')
print(f'\nFinal OOF with thresholds: {final_oof_f1:.4f}')
print(f'Thresholds: {best_thresholds}')
print(classification_report(y, final_oof_preds, digits=4))

In [ ]:
# =============================================================================
# Cell 10: Clinical Guardrails + Submission
# =============================================================================

test_preds = apply_thresholds(te_blend, best_thresholds)
test['target'] = test_preds

def apply_clinical_guardrails(row):
    text = str(row['report']).lower()
    pred = int(row['target'])
    
    birads_match = re.search(r'(?:bi-?rads|categoria)\s*:?\s*(\d)', text)
    if birads_match:
        mentioned = int(birads_match.group(1))
        if 0 <= mentioned <= 6:
            return mentioned
    
    if re.search(r'resultado de cine grau 3|carcinoma|\bcdis\b|neoplasia maligna', text):
        return 6
    
    if re.search(r'n[oó]dulo\s+espiculad', text) and pred < 4:
        return 5
    if 'espiculad' in text and 'distor' in text and pred < 4:
        return 5
    
    if re.search(r'calcifica[cç][aã]o grosseira|calcifica[cç][aã]o vascular', text):
        if pred >= 4 and not re.search(r'espiculad|suspeit|malign|distor', text):
            return 2
    
    return pred

test['target'] = test.apply(apply_clinical_guardrails, axis=1)

submission = test[['ID', 'target']].copy()
submission['target'] = submission['target'].astype(int)
submission.to_csv('submission.csv', index=False)

print('Prediction distribution:')
print(submission['target'].value_counts().sort_index())
print(f'\nSubmission saved! Shape: {submission.shape}')
print(submission)